<!-- Cell 1 -->
Deliverable: unified, cleaned dataset

In [ ]:
# Cell 2
import pandas as pd
import numpy as np
import re
import html

In [ ]:
# Cell 3
from pathlib import Path
import sys

# Get the current working directory
PROJECT_ROOT = Path.cwd()

# If running from the notebooks folder, go up one level
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add project root to Python's import path
sys.path.insert(0, str(PROJECT_ROOT))

# Import loader
from src.loader import load_csv_folder

# Dataset directory
ZENODO_DIR = PROJECT_ROOT / "data" / "raw" / "zenodo"

# put all csv files into single df
data_df = load_csv_folder(ZENODO_DIR)

#display data
data_df

In [ ]:
# Cell 4
print(data_df.columns.tolist())
print(data_df.info())

In [ ]:
# Cell 5
# stamdardize case and remove whitespace from column names
data_df.columns = (
    data_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [ ]:
# Cell 6
#missing values
print(data_df[["subject", "body"]].isna().sum())

In [ ]:
# Cell 7
#replace missing with empty string
data_df["subject"] = data_df["subject"].fillna("")
data_df["body"] = data_df["body"].fillna("")

In [ ]:
# Cell 8
missing_content = (
    (data_df["subject"].str.strip() == "") &
    (data_df["body"].str.strip() == "")
)

print("Emails with no subject or body:", missing_content.sum())

<!-- Cell 9 -->
### Flagged Data to eliminate

In [ ]:
# Cell 10
flagged_df = data_df[missing_content]
flagged_df.head() 

In [ ]:
# Cell 11
#add flagged to flagged.csv
import os

file_path = "../data/flagged/flagged.csv"
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# write the header if the file does not exist yet
file_exists = os.path.isfile(file_path)

flagged_df.to_csv(
    file_path,
    mode="a",
    header=not file_exists,
    index=False
)

<!-- Cell 12 -->
#### Decode HTML

Email bodies may contain HTML marks and entities. 

In [ ]:
# Cell 13
# before decode
def contains_html(text):
    if pd.isna(text):
        return False

    pattern = r"</?(html|body|head|div|p|br|a|span|table|tr|td|img|style|font|meta|title|h[1-6])\b[^>]*>"

    return bool(re.search(pattern, str(text), re.IGNORECASE))


html_emails = data_df[data_df["body"].apply(contains_html)]

print("Emails containing possible HTML:", len(html_emails))

display(html_emails[["subject", "body", "source_file"]].head(10))

In [ ]:
# Cell 14
def clean_email_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # decode HTML entities (including nested encoding)
    while True:
        decoded = html.unescape(text)

        if decoded == text:
            break

        text = decoded

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
# Cell 15
data_df["subject"] = data_df["subject"].apply(clean_email_text)
data_df["body"] = data_df["body"].apply(clean_email_text)

In [ ]:
# Cell 16
data_df['body']

In [ ]:
# Cell 17
#check if html remains
#tags
html_remaining = data_df[
    data_df["body"].apply(contains_html)
]

print("Emails with remaining possible HTML tags:", len(html_remaining))


#html entities
entity_pattern = r"&(?:amp|lt|gt|quot|apos|nbsp);|&#\d+;|&#x[0-9a-fA-F]+;"

entity_remaining = data_df[
    data_df["body"].str.contains(
        entity_pattern,
        case=False,
        regex=True,
        na=False
    )
]

print("Emails with remaining HTML entities:", len(entity_remaining))

<!-- Cell 18 -->
## Text processing
lowercase / normalize text

In [ ]:
# Cell 19
print(data_df.dtypes)

<!-- Cell 20 -->
.fillna('') – Replaces missing values with empty strings.



.astype(str) – Converts values into strings.



.str.lower() – Converts text to lowercase.



.str.strip() – Removes leading and trailing whitespace.

In [ ]:
# Cell 21
#lowercase / strip whitespace / normalize
#will keep others preserved
columns = ['body', 'subject', 'sender', 'receiver']

for col in columns:
    data_df[col] = data_df[col].fillna("").astype(str).str.lower().str.strip()

#convert date
data_df['date'] = pd.to_datetime(
    data_df['date'],
    errors='coerce',
    utc=True
)

<!-- Cell 22 -->
#### URL feature extraction

In [ ]:
# Cell 23
#make URL a string type
data_df['urls'] = (
    data_df['urls']
    .fillna('')
    .astype(str)
)

In [ ]:
# Cell 24
data_df['urls'].unique()

<!-- Cell 25 -->
The `urls` column mixes formats: `0`/`1`, `0.0`/`1.0`, and actual link lists (only Nazario_5). Enron and Ling have no URL info.

Steps:
1. **`url_flag`**: one consistent 0/1 column (NaN where the file has no URL info)
2. **`url_list`**: the actual links, found in each email's subject and body
3. **URL features**: numbers built from those links that a model can use

In [ ]:
# Cell 26
# make the urls column one consistent format
# 1 = email has links, 0 = no links, NaN = file has no url info (Enron, Ling)
def normalize_url_flag(value):
    value = str(value).strip()
    if value in ("1", "1.0"):
        return 1
    if value in ("0", "0.0"):
        return 0
    if value.startswith("["):        # Nazario_5 stores the actual links as a list
        return 0 if value == "[]" else 1
    return np.nan                    # empty = no url info for this file


data_df["url_flag"] = data_df["urls"].apply(normalize_url_flag)
data_df["url_flag"].value_counts(dropna=False)

In [ ]:
# Cell 27
# find the links written in each email (subject + body)
# only Nazario_5 lists its actual links, so we pull them out of the text for every file
import ast

URL_PATTERN = re.compile(r"(?:https?://|www\.)[^\s<>\"'()\[\]{}]+")


def find_urls(subject, body, urls):
    found = URL_PATTERN.findall(subject + " " + body)

    # add the links Nazario_5 already listed, e.g. "['http://a.com', 'http://b.com']"
    if urls.startswith("["):
        try:
            found += [str(u) for u in ast.literal_eval(urls)]
        except (ValueError, SyntaxError):
            pass

    # remove punctuation stuck to the end, lowercase, drop repeats (keeps order)
    cleaned = [u.rstrip(".,;:!?").lower() for u in found]
    return list(dict.fromkeys(cleaned))


data_df["url_list"] = [
    find_urls(s, b, u)
    for s, b, u in zip(data_df["subject"], data_df["body"], data_df["urls"])
]

data_df.loc[data_df["url_list"].str.len() > 0, ["url_list", "label"]].head()

In [ ]:
# Cell 28
# build number features from each email's list of links
from urllib.parse import urlparse

SUSPICIOUS_WORDS = ["login", "signin", "verify", "account", "update",
                    "secure", "bank", "confirm", "password", "paypal"]
SHORTENERS = {"bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly", "is.gd", "buff.ly"}
IP_PATTERN = re.compile(r"^\d{1,3}(?:\.\d{1,3}){3}$")   # e.g. 192.168.0.1


def get_domain(url):
    # "http://www.paypal.com/login" -> "www.paypal.com"
    if url.startswith("www."):
        url = "http://" + url
    try:
        return urlparse(url).hostname or ""
    except ValueError:
        return ""


def url_features(urls):
    domains = [get_domain(u) for u in urls]
    return {
        "url_count": len(urls),                                                      # number of links
        "url_unique_domains": len({d for d in domains if d}),                        # different websites linked
        "url_max_length": max((len(u) for u in urls), default=0),                    # longest link
        "url_has_ip": int(any(IP_PATTERN.match(d) for d in domains)),                # link uses an IP instead of a name
        "url_has_http_only": int(any(u.startswith("http://") for u in urls)),        # link without https
        "url_has_at_symbol": int(any("@" in u for u in urls)),                       # "@" can hide the real site
        "url_has_shortener": int(any(d.removeprefix("www.") in SHORTENERS for d in domains)),  # bit.ly etc.
        "url_has_suspicious_word": int(any(w in u for u in urls for w in SUSPICIOUS_WORDS)),   # login, verify...
        "url_max_subdomains": max((d.count(".") for d in domains), default=0),       # dots in the website name
    }


url_feats = pd.DataFrame([url_features(u) for u in data_df["url_list"]], index=data_df.index)
url_feature_cols = list(url_feats.columns)

data_df = pd.concat([data_df, url_feats], axis=1)

# turn the lists back into text (links separated by spaces)
# so drop_duplicates() and saving to CSV work later
data_df["url_list"] = data_df["url_list"].str.join(" ")

data_df[url_feature_cols].describe().round(2)

In [ ]:
# Cell 29
# check: does the original url flag agree with the links we found?
print(pd.crosstab(data_df["url_flag"], data_df["url_count"] > 0, dropna=False))

# average of each URL feature for legit (0) vs phishing/spam (1)
data_df.groupby("label")[url_feature_cols].mean().round(3).T

<!-- Cell 30 -->
### Eliminate flagged rows

- duplicates
- missing values

In [ ]:
# Cell 31
#remove flagged from main df

#remove duplicate rows
data_df = data_df.drop_duplicates()



#drop the rows where missing_content is True
data_df = data_df[~missing_content]


<!-- Cell 32 -->
#### New cleaned dataset

In [ ]:
# Cell 33
#preserve original
print(data_df["source_file"].value_counts())
print("\n\nFinal shape:", data_df.shape)

In [ ]:
# Cell 34
#exporting cleaned dataset
import os

os.makedirs("../data/processed", exist_ok=True)
data_df.to_csv(
    "../data/processed/cleaned_emails.csv",
    index=False
)